# This challenge adds another call to translate the Brochure to spanish.



In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from modified_scrapper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [2]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()


API key looks good so far


In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [5]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links


In [6]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [7]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [8]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [9]:
def stream_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ]
    )
    result = response.choices[0].message.content

    stream = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": "Translate the following marketing brochure to Spanish: " + result}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [11]:
# If you get a `Key` error when calling the function below,
# it means that the scrapper was not able to find any relevant links on the website, try another one
stream_brochure("MDB Capital", "https://www.mdb.com/")


Selecting relevant links for https://www.mdb.com/ by calling gpt-5-nano
Found 20 relevant links


MDB Capital: Lanzamos Grandes Ideas

MDB Capital combina visión de venture, innovación en corretaje y una mentalidad inclusiva y de horizonte largo para convertir grandes ideas en valor duradero. Nuestra plataforma está diseñada para ser más democrática, líquida y eficiente para inventores, universidades, cofundadores, inversionistas y empleados por igual.

Temas clave que debes conocer
- Fundamento de Public Venture: un modelo que alinea intereses entre inventores, universidades, cofundadores, inversionistas y la sociedad, enfatizando una mayor participación accionaria, la colaboración basada en la confianza y el valor a largo plazo.
- Plataforma enfocada en venture: la primera correduría y plataforma en línea construidas específicamente para la inversión en etapas de venture. MDB Direct ofrece servicios de corretaje con liquidación propia y una experiencia de negociación adaptada a oportunidades de venture.
- PI y apoyo estratégico: PatentVest complementa a MDB Capital al proporcionar soluciones integradas de PI y desarrollo de estrategias para guiar a las empresas tecnológicas a través de la ley de PI y maximizar su valor.

Quiénes somos
- MDB Capital es la marca detrás de MDB Capital Holdings (NASDAQ: MDBH) y sus filiales. Operamos como Public Ventures, LLC, haciendo negocios como MDB Capital, y como MDB Capital Brokerage con la plataforma MDB Direct.
- Nuestra plataforma y servicios están diseñados para inversionistas y titulares de cuentas de MDB, con membresía en FINRA y SIPC y una base registrada como corredora y distribuidora de valores.

Qué hacemos para inventores, fundadores y empresas
- Marco Public Venture: un enfoque holístico que enfatiza dinámicas de inversión democráticas, líquidas y eficientes para apoyar el progreso de invención al mercado.
- Propiedad y creación de valor: las empresas generan un valor significativo para los fundadores gracias a participaciones de propiedad, usualmente superiores a las del venture tradicional.
- Crecimiento impulsado por PI: PatentVest se asocia con MDB Capital para ofrecer desarrollo de estrategia de PI y servicios de PI que ayudan a las empresas tecnológicas a navegar la ley de PI y maximizar su valor.

Qué hacemos para inversores
- Comunidad de Inversionistas MDB: una red confiable y colaborativa diseñada para equilibrar riesgo y rendimiento con un enfoque impulsado por la comunidad.
- Liquidez y acceso: una plataforma que busca ofrecer un potencial de rendimiento sustancial sin la iliquidez que suele asociarse a fondos de venture tradicionales.
- Construcción de riqueza a largo plazo: el modelo MDB ha creado oportunidades de riqueza para nuestros empleados a largo plazo y apoya un impacto de sociedad a sociedad a través de inversiones más inteligentes y con propósito.

Cultura, personas y carreras
- Cultura: una comunidad basada en la confianza y con un propósito claro, donde la colaboración y el pensamiento a largo plazo son fundamentales. Nuestro enfoque enfatiza resultados compartidos y un sentido de pertenencia dentro de un ecosistema vibrante de inversores e inventores.
- Empleados: la plataforma y el modelo MDB han contribuido a una creación de riqueza significativa y de largo plazo para nuestro equipo, subrayando un enfoque en el crecimiento profesional y la generación de valor.
- Carreras y unirse al equipo: el sitio destaca una ruta de “Únete a nosotros” y las páginas del equipo directivo, lo que señala oportunidades para contribuir a una plataforma pionera. Aunque no se listan roles específicos en el material proporcionado, MDB Capital invita a candidatos interesados a explorar oportunidades en el sitio.

Cómo participar y por dónde empezar
- Enviar Tu Gran Idea: una llamada a la acción central para inventores y fundadores para involucrarse con MDB Capital.
- Abrir una cuenta o aprender sobre MDB Direct: para inversionistas que buscan acceso centrado en venture y una plataforma de negociación dedicada.
- Suscríbase a nuestra lista de correo: manténgase informado con la información, ideas y noticias más recientes de MDB Capital.
- Explorar contenido mediático y educativo: videos, informes y artículos que proporcionan contexto sobre nuestra plataforma, la creación de valor y nuestro enfoque.

Fundamentos regulatorios y de gobernanza
- MDB Capital opera como una corredora y distribuidora de valores registrada, miembro de FINRA y SIPC, con acceso a BrokerCheck para diligencia debida y transparencia.
- La gobernanza, la continuidad del negocio y las revelaciones forman parte de las secciones del sitio para la protección continua de inversores y clientes.

PI y servicios estratégicos
- PatentVest complementa la plataforma de MDB como solución de PI y proveedor de servicios, guiando a las empresas tecnológicas en el desarrollo de estrategias de PI y las complejidades legales relacionadas.

Referencias rápidas sobre la estructura del sitio
- Página de aterrizaje: MDB Capital – Lanzamos Grandes Ideas; secciones sobre Por qué Public Venture, Qué es Public Venture, Ventaja de Public Venture, Cómo Invertir, Por qué MDB, Diferencia MDB, Criterios de Inversión, Proceso de Curación, Capacidades de Lanzamiento, MDB Capital Brokerage, Comunidad, MDB Direct, IRAs, Abrir una Cuenta, Medios, Liderazgo, Enviar Tu Gran Idea, Inversores MDBH, Información de Acciones, Gobernanza, Alertas, Eventos y más.
- Acerca de / Nuestra Plataforma: Explica a MDB Capital como el paraguas de la plataforma, la correduría enfocada en venture, el modelo Public Ventures y la integración con servicios de PI a través de PatentVest.

Conclusión
MDB Capital está diseñado para lanzar grandes ideas al combinar un enfoque público-venture democrático y líquido con una plataforma de negociación centrada en venture y un sólido apoyo en PI. Nuestro objetivo es alinear incentivos entre inventores, universidades, inversionistas, empleados y la sociedad en general, creando valor duradero mediante la colaboración, la propiedad y el pensamiento a largo plazo. Si eres un inventor con un avance innovador, una universidad que busca comercializar investigación, un inversionista que busca exposición en etapas de venture con una comunidad, o un profesional que desea contribuir a una plataforma pionera, MDB Capital te invita a conectarte a través de Enviar Tu Gran Idea, a abrir una cuenta o a unirte a nuestra lista de correo.